# Plot distributions

In [ ]:
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns

## Load data

In [ ]:

df_eqgat = pd.read_csv("predictions/unconditional/eqgat/eqgat_100000_predictions.csv")
df_eqgat["method"] = "EQGAT"
df_gcdm = pd.read_csv("predictions/unconditional/gcdm/gcdm_100000_predictions.csv")
df_gcdm["method"] = "GCDM-SBDD"
# df_geoldm = pd.read_csv("predictions/unconditional/geoldm/geoldm_100000_predictions.csv")
# df_geoldm["method"] = "GeoLDM"
df_semla = pd.read_csv("predictions/unconditional/semlaflow/semlaflow_100000_predictions.csv")
df_semla["method"] = "SemlaFlow"
df_flowmol = pd.read_csv("predictions/unconditional/molflow/molflow_100000_predictions.csv")[:100000]
df_flowmol["method"] = "MolFlow"

df_train = pd.read_csv("data/unconditional/geom-drugs/train.csv")
df_train["method"] = "GEOM Drugs Training"
# df_val = pd.read_csv("predictions_data/val.csv")
# df_val["method"] = "GEOM Drugs Validation"
# df_test = pd.read_csv("predictions_data/test.csv")
# df_test["method"] = "GEOM Drugs Testing"

df = pd.concat([df_flowmol, df_semla, df_eqgat, df_train])
df = df[~df.fail.fillna(0).astype(bool)] # Evaluation script does not break molecules correctly, introducting these rows

df["total"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

# Create valid table

In [ ]:
df_valid = df[['method',"total", 'connected', 'chemical', 'physical', "valid"]].groupby("method").sum()
# df_valid.columns = ["generated", "valid"]
df_valid

In [ ]:
df[["method", "num_heavy"]].groupby("method").max().astype(int)

## Name metrics

In [ ]:
metrics = {
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "Quantitative Estimation of Drug-likeness",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

## Plot one

In [ ]:
# seaborn

metric = "logp"  # lipophilicity
metric = "sa"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 10)

In [ ]:
# seaborn

metric = "logp"  # lipophilicity
metric = "sa"
metric = "spacial"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 120)

In [ ]:
# seaborn

metric = "logp"  # lipophilicity
metric = "sa"
metric = "qed"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 1)

In [ ]:
# seaborn

metric = "energy_ratio"
name = "Energy Ratio"
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=100,
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    log_scale=True if metric == "energy_ratio" else False,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0.5, 200)
# plt.savefig(f"plots/plot_{metric}.png")
# plt.close()

In [ ]:
# seaborn

metric = "num_heavy"
name = metrics[metric]
sns.histplot(
    df[df.valid][["method", metric]].reset_index(drop=True),
    x=metric,
    bins=np.array(range(int(df[df.valid]["num_heavy"].max()) + 1)),
    hue="method",
    cumulative=False,
    common_norm=False,
    stat="density",
    element="step",
    fill=True,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0, 60)

## Plot and save all

In [ ]:
for metric, name in tqdm(metrics.items()):
    sns.histplot(
        df[df.valid][["method", metric]].reset_index(drop=True),
        x=metric,
        hue="method",
        cumulative=False,
        common_norm=False,
        stat="density",
        element="step",
        # legend=True, palette="tab10", linewidth=1.5
    )
    plt.title(name)
    plt.xlabel(name)
    plt.savefig(f"plots/unconditonal/plot_{metric}.png")
    plt.close()

# OLD

In [ ]:
sns.histplot(
    df[df.method.str.startswith("GEOM")][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    cumulative=False,
    fill=False,
    common_norm=False,
    stat="density",
    element="step",
    log_scale=True if metric == "energy_ratio" else False,
    # legend=True, palette="tab10", linewidth=1.5
)
sns.histplot(
    df[~df.method.str.startswith("GEOM")][["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    cumulative=False,
    fill=True,
    common_norm=False,
    stat="density",
    element="step",
    log_scale=True if metric == "energy_ratio" else False,
    # legend=True, palette="tab10", linewidth=1.5
)
plt.title(name)
plt.xlabel(name)
plt.xlim(0.5, 20)
plt.show()